# sesac_power_station_4 MySQL CSV loader

This notebook connects to MySQL, creates the ERD v2 tables, loads the CSV files in `dalykit/data/erd_v2_csv`, and verifies that the data was inserted.

Run from top to bottom.

What this notebook does:

1. Finds the project folder and CSV folder.
2. Connects to MySQL and creates the database if needed.
3. Creates the ERD v2 tables.
4. Checks that every `tb_*.csv` header matches the table schema.
5. Loads CSV files in foreign-key order.
6. Confirms MySQL row counts match CSV row counts.
7. Checks foreign keys, sensor duplicate keys, and sample rows.

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import pymysql
from IPython.display import display

# Find the project root even if Jupyter starts from a nearby folder.
cwd = Path.cwd().resolve()
PROJECT_ROOT = None
candidates = [cwd, *cwd.parents]
if cwd.exists():
    candidates.extend(path for path in cwd.iterdir() if path.is_dir())

for candidate in candidates:
    if (candidate / 'dalykit' / 'data').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError('Project root not found. Run this notebook inside the final_project workspace.')

CSV_DIR = PROJECT_ROOT / 'dalykit' / 'data' / 'erd_v2_csv'
if not CSV_DIR.exists():
    raise FileNotFoundError(f'CSV folder not found: {CSV_DIR}')

DB_CONFIG = {
    'host': '127.0.0.1',
    'port': 3306,
    'user': 'root',
    'password': 'test1234',
    'database': 'sesac_power_station_db',
    'charset': 'utf8mb4',
    'local_infile': True,
}

# Recreate tables so rerunning the notebook starts from a clean DB state.
RECREATE_TABLES = True

# True uses MySQL LOAD DATA LOCAL INFILE first. If MySQL blocks it,
# the loader automatically falls back to pandas chunk inserts.
USE_LOAD_DATA = True

# Chunk insert fallback is much slower than LOAD DATA, especially for
# tb_prediction.csv and the sensor CSVs.
CHUNKSIZE = 50_000
SAMPLE_ROWS = 5

print('PROJECT_ROOT =', PROJECT_ROOT)
print('CSV_DIR =', CSV_DIR)
print('CSV files =', sorted(path.name for path in CSV_DIR.glob('tb_*.csv')))

PROJECT_ROOT = C:\Users\Owner\Final_Project\final_project
CSV_DIR = C:\Users\Owner\Final_Project\final_project\dalykit\data\erd_v2_csv
CSV files = ['tb_alarm_info.csv', 'tb_bac_sensor.csv', 'tb_daily_report.csv', 'tb_dgan_sensor.csv', 'tb_mac_a_sensor.csv', 'tb_mac_b_sensor.csv', 'tb_model_info.csv', 'tb_part.csv', 'tb_power_plant.csv', 'tb_power_station.csv', 'tb_prediction.csv', 'tb_user_info.csv', 'tb_vhp_sensor.csv']


## 1. Connect to MySQL

Change only `DB_CONFIG` above if your local MySQL user, password, or database name is different.

In [2]:
def get_conn(database=True, autocommit=False):
    config = DB_CONFIG.copy()
    if not database:
        config.pop('database', None)
    return pymysql.connect(**config, autocommit=autocommit)


# Create the database before connecting to it directly.
conn = get_conn(database=False, autocommit=True)
try:
    with conn.cursor() as cur:
        cur.execute(
            f"CREATE DATABASE IF NOT EXISTS `{DB_CONFIG['database']}` "
            "CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci"
        )
finally:
    conn.close()

conn = get_conn()
try:
    with conn.cursor() as cur:
        cur.execute('SELECT DATABASE(), VERSION()')
        print(cur.fetchone())
finally:
    conn.close()

('sesac_power_station_db', '8.0.45')


## 2. Check fast CSV loading

The fastest path uses `LOAD DATA LOCAL INFILE`. If MySQL reports `local_infile = OFF`, this notebook can still load data with chunk inserts, but it will take much longer.

In [3]:
conn = get_conn()
try:
    with conn.cursor() as cur:
        cur.execute("SHOW VARIABLES LIKE 'local_infile'")
        local_infile_status = cur.fetchone()
        print('local_infile =', local_infile_status)
finally:
    conn.close()

if local_infile_status and str(local_infile_status[1]).upper() != 'ON':
    print('LOAD DATA LOCAL INFILE may be blocked. The notebook will fall back to chunk inserts if needed.')

local_infile = ('local_infile', 'OFF')
LOAD DATA LOCAL INFILE may be blocked. The notebook will fall back to chunk inserts if needed.


## 3. Define tables and CSV columns

The column order below must match both the MySQL table schema and each CSV header.

In [4]:
TABLE_COLUMNS = {
    'tb_power_station': ['station_id', 'station_name', 'station_lat', 'station_lon', 'station_add', 'station_capacity'],
    'tb_power_plant': ['plant_id', 'station_id', 'plant_name', 'fuel_type', 'plant_status'],
    'tb_part': ['part_id', 'plant_id', 'part_type', 'part_name'],
    'tb_user_info': ['user_num', 'email', 'password_hash', 'name', 'permission', 'created_at', 'assigned_station'],
    'tb_model_info': ['model_id', 'part_id', 'model_name', 'model_version', 'threshold'],
    'tb_mac_a_sensor': ['mac_a_id', 'part_id', 'measured_at', 'current', 'NDE_temp', 'DE_temp', 'NDE_X_vibration', 'NDE_Y_vibration', 'DE_X_vibration', 'DE_Y_vibration'],
    'tb_mac_b_sensor': ['mac_b_id', 'part_id', 'measured_at', 'current', 'NDE_temp', 'DE_temp', 'NDE_X_vibration', 'NDE_Y_vibration', 'DE_X_vibration', 'DE_Y_vibration'],
    'tb_bac_sensor': ['bac_id', 'part_id', 'measured_at', 'current', 'NDE_temp', 'DE_temp', 'NDE_X_vibration', 'NDE_Y_vibration', 'DE_X_vibration', 'DE_Y_vibration'],
    'tb_dgan_sensor': ['dgan_id', 'part_id', 'measured_at', 'current', 'NDE_temp', 'DE_temp', 'NDE_X_vibration', 'NDE_Y_vibration', 'DE_X_vibration', 'DE_Y_vibration'],
    'tb_vhp_sensor': ['vhp_id', 'part_id', 'measured_at', 'current', 'NDE_temp', 'DE_temp', 'NDE_X_vibration', 'NDE_Y_vibration', 'DE_X_vibration', 'DE_Y_vibration'],
    'tb_prediction': [
        'prediction_id', 'model_id', 'anomaly_score', 'prediction_time',
        'current_recon_error', 'NDE_temp_recon_error', 'DE_temp_recon_error',
        'NDE_X_vib_recon_error', 'NDE_Y_vib_recon_error',
        'DE_X_vib_recon_error', 'DE_Y_vib_recon_error',
    ],
    'tb_alarm_info': ['alarm_id', 'user_num', 'prediction_id', 'alarm_content', 'alarm_type', 'alarm_create_at', 'anomaly_score'],
    'tb_daily_report': ['report_id', 'alarm_id', 'report_created_at', 'issue_content', 'action_detail'],
}

LOAD_ORDER = [
    'tb_power_station',
    'tb_power_plant',
    'tb_part',
    'tb_user_info',
    'tb_model_info',
    'tb_mac_a_sensor',
    'tb_mac_b_sensor',
    'tb_bac_sensor',
    'tb_dgan_sensor',
    'tb_vhp_sensor',
    'tb_prediction',
    'tb_alarm_info',
    'tb_daily_report',
]

SENSOR_TABLES = ['tb_mac_a_sensor', 'tb_mac_b_sensor', 'tb_bac_sensor', 'tb_dgan_sensor', 'tb_vhp_sensor']
CSV_FILES = {table: CSV_DIR / f'{table}.csv' for table in LOAD_ORDER}

DROP_TABLES_SQL = [
    'DROP TABLE IF EXISTS `tb_daily_report`',
    'DROP TABLE IF EXISTS `tb_alarm_info`',
    'DROP TABLE IF EXISTS `tb_prediction`',
    'DROP TABLE IF EXISTS `tb_model_info`',
    'DROP TABLE IF EXISTS `tb_mac_a_sensor`',
    'DROP TABLE IF EXISTS `tb_mac_b_sensor`',
    'DROP TABLE IF EXISTS `tb_bac_sensor`',
    'DROP TABLE IF EXISTS `tb_dgan_sensor`',
    'DROP TABLE IF EXISTS `tb_vhp_sensor`',
    'DROP TABLE IF EXISTS `tb_part`',
    'DROP TABLE IF EXISTS `tb_power_plant`',
    'DROP TABLE IF EXISTS `tb_user_info`',
    'DROP TABLE IF EXISTS `tb_power_station`',
]

CREATE_TABLES_SQL = [
    """
    CREATE TABLE IF NOT EXISTS `tb_power_station` (
      `station_id` INT NOT NULL AUTO_INCREMENT,
      `station_name` VARCHAR(255) NOT NULL,
      `station_lat` DOUBLE NULL,
      `station_lon` DOUBLE NULL,
      `station_add` VARCHAR(255) NOT NULL,
      `station_capacity` DOUBLE NOT NULL,
      CONSTRAINT `PK_TB_POWER_STATION` PRIMARY KEY (`station_id`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
    """,
    """
    CREATE TABLE IF NOT EXISTS `tb_power_plant` (
      `plant_id` INT NOT NULL AUTO_INCREMENT,
      `station_id` INT NOT NULL,
      `plant_name` VARCHAR(255) NOT NULL,
      `fuel_type` VARCHAR(255) NOT NULL,
      `plant_status` VARCHAR(255) NOT NULL,
      CONSTRAINT `PK_TB_POWER_PLANT` PRIMARY KEY (`plant_id`),
      CONSTRAINT `FK_POWER_PLANT_STATION`
        FOREIGN KEY (`station_id`) REFERENCES `tb_power_station` (`station_id`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
    """,
    """
    CREATE TABLE IF NOT EXISTS `tb_part` (
      `part_id` INT NOT NULL AUTO_INCREMENT,
      `plant_id` INT NOT NULL,
      `part_type` VARCHAR(255) NOT NULL COMMENT 'mac a,b,bac',
      `part_name` VARCHAR(255) NOT NULL COMMENT '1??_MAC_A',
      CONSTRAINT `PK_TB_PART` PRIMARY KEY (`part_id`),
      CONSTRAINT `FK_PART_PLANT`
        FOREIGN KEY (`plant_id`) REFERENCES `tb_power_plant` (`plant_id`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
    """,
    """
    CREATE TABLE IF NOT EXISTS `tb_user_info` (
      `user_num` INT NOT NULL AUTO_INCREMENT COMMENT '?? ?? ?? ?? ??',
      `email` VARCHAR(255) NOT NULL COMMENT '??? ???',
      `password_hash` VARCHAR(255) NOT NULL COMMENT '??? ????',
      `name` VARCHAR(255) NOT NULL COMMENT '??? ??',
      `permission` VARCHAR(255) NOT NULL COMMENT '???, ???',
      `created_at` DATETIME NULL,
      `assigned_station` VARCHAR(255) NULL,
      CONSTRAINT `PK_TB_USER_INFO` PRIMARY KEY (`user_num`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
    """,
    """
    CREATE TABLE IF NOT EXISTS `tb_model_info` (
      `model_id` INT NOT NULL AUTO_INCREMENT,
      `part_id` INT NOT NULL COMMENT '?? ?? ?? ??',
      `model_name` VARCHAR(255) NOT NULL COMMENT '1??_MAC_A ??',
      `model_version` VARCHAR(255) NULL COMMENT 'v1',
      `threshold` DOUBLE NOT NULL,
      CONSTRAINT `PK_TB_MODEL_INFO` PRIMARY KEY (`model_id`),
      CONSTRAINT `FK_MODEL_INFO_PART`
        FOREIGN KEY (`part_id`) REFERENCES `tb_part` (`part_id`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
    """,
    """
    CREATE TABLE IF NOT EXISTS `tb_mac_a_sensor` (
      `mac_a_id` BIGINT NOT NULL AUTO_INCREMENT,
      `part_id` INT NOT NULL COMMENT '?? ?? ?? ??',
      `measured_at` DATETIME NOT NULL,
      `current` DOUBLE NULL,
      `NDE_temp` DOUBLE NULL,
      `DE_temp` DOUBLE NULL,
      `NDE_X_vibration` DOUBLE NULL,
      `NDE_Y_vibration` DOUBLE NULL,
      `DE_X_vibration` DOUBLE NULL,
      `DE_Y_vibration` DOUBLE NULL,
      CONSTRAINT `PK_TB_MAC_A_SENSOR` PRIMARY KEY (`mac_a_id`),
      CONSTRAINT `FK_MAC_A_SENSOR_PART`
        FOREIGN KEY (`part_id`) REFERENCES `tb_part` (`part_id`),
      CONSTRAINT `UK_MAC_A_SENSOR_PART_TIME` UNIQUE (`part_id`, `measured_at`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
    """,
    """
    CREATE TABLE IF NOT EXISTS `tb_mac_b_sensor` (
      `mac_b_id` BIGINT NOT NULL AUTO_INCREMENT,
      `part_id` INT NOT NULL COMMENT '?? ?? ?? ??',
      `measured_at` DATETIME NOT NULL,
      `current` DOUBLE NULL,
      `NDE_temp` DOUBLE NULL,
      `DE_temp` DOUBLE NULL,
      `NDE_X_vibration` DOUBLE NULL,
      `NDE_Y_vibration` DOUBLE NULL,
      `DE_X_vibration` DOUBLE NULL,
      `DE_Y_vibration` DOUBLE NULL,
      CONSTRAINT `PK_TB_MAC_B_SENSOR` PRIMARY KEY (`mac_b_id`),
      CONSTRAINT `FK_MAC_B_SENSOR_PART`
        FOREIGN KEY (`part_id`) REFERENCES `tb_part` (`part_id`),
      CONSTRAINT `UK_MAC_B_SENSOR_PART_TIME` UNIQUE (`part_id`, `measured_at`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
    """,
    """
    CREATE TABLE IF NOT EXISTS `tb_bac_sensor` (
      `bac_id` BIGINT NOT NULL AUTO_INCREMENT,
      `part_id` INT NOT NULL COMMENT '?? ?? ?? ??',
      `measured_at` DATETIME NOT NULL,
      `current` DOUBLE NULL,
      `NDE_temp` DOUBLE NULL,
      `DE_temp` DOUBLE NULL,
      `NDE_X_vibration` DOUBLE NULL,
      `NDE_Y_vibration` DOUBLE NULL,
      `DE_X_vibration` DOUBLE NULL,
      `DE_Y_vibration` DOUBLE NULL,
      CONSTRAINT `PK_TB_BAC_SENSOR` PRIMARY KEY (`bac_id`),
      CONSTRAINT `FK_BAC_SENSOR_PART`
        FOREIGN KEY (`part_id`) REFERENCES `tb_part` (`part_id`),
      CONSTRAINT `UK_BAC_SENSOR_PART_TIME` UNIQUE (`part_id`, `measured_at`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
    """,
    """
    CREATE TABLE IF NOT EXISTS `tb_dgan_sensor` (
      `dgan_id` BIGINT NOT NULL AUTO_INCREMENT,
      `part_id` INT NOT NULL COMMENT '?? ?? ?? ??',
      `measured_at` DATETIME NOT NULL,
      `current` DOUBLE NULL,
      `NDE_temp` DOUBLE NULL,
      `DE_temp` DOUBLE NULL,
      `NDE_X_vibration` DOUBLE NULL,
      `NDE_Y_vibration` DOUBLE NULL,
      `DE_X_vibration` DOUBLE NULL,
      `DE_Y_vibration` DOUBLE NULL,
      CONSTRAINT `PK_TB_DGAN_SENSOR` PRIMARY KEY (`dgan_id`),
      CONSTRAINT `FK_DGAN_SENSOR_PART`
        FOREIGN KEY (`part_id`) REFERENCES `tb_part` (`part_id`),
      CONSTRAINT `UK_DGAN_SENSOR_PART_TIME` UNIQUE (`part_id`, `measured_at`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
    """,
    """
    CREATE TABLE IF NOT EXISTS `tb_vhp_sensor` (
      `vhp_id` BIGINT NOT NULL AUTO_INCREMENT,
      `part_id` INT NOT NULL COMMENT '?? ?? ?? ??',
      `measured_at` DATETIME NOT NULL,
      `current` DOUBLE NULL,
      `NDE_temp` DOUBLE NULL,
      `DE_temp` DOUBLE NULL,
      `NDE_X_vibration` DOUBLE NULL,
      `NDE_Y_vibration` DOUBLE NULL,
      `DE_X_vibration` DOUBLE NULL,
      `DE_Y_vibration` DOUBLE NULL,
      CONSTRAINT `PK_TB_VHP_SENSOR` PRIMARY KEY (`vhp_id`),
      CONSTRAINT `FK_VHP_SENSOR_PART`
        FOREIGN KEY (`part_id`) REFERENCES `tb_part` (`part_id`),
      CONSTRAINT `UK_VHP_SENSOR_PART_TIME` UNIQUE (`part_id`, `measured_at`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
    """,
    """
    CREATE TABLE IF NOT EXISTS `tb_prediction` (
      `prediction_id` INT NOT NULL AUTO_INCREMENT,
      `model_id` INT NOT NULL,
      `anomaly_score` DOUBLE NOT NULL,
      `prediction_time` DATETIME NOT NULL,
      `current_recon_error` DOUBLE NOT NULL,
      `NDE_temp_recon_error` DOUBLE NOT NULL,
      `DE_temp_recon_error` DOUBLE NOT NULL,
      `NDE_X_vib_recon_error` DOUBLE NOT NULL,
      `NDE_Y_vib_recon_error` DOUBLE NOT NULL,
      `DE_X_vib_recon_error` DOUBLE NOT NULL,
      `DE_Y_vib_recon_error` DOUBLE NOT NULL,
      CONSTRAINT `PK_TB_PREDICTION` PRIMARY KEY (`prediction_id`),
      CONSTRAINT `FK_PREDICTION_MODEL`
        FOREIGN KEY (`model_id`) REFERENCES `tb_model_info` (`model_id`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
    """,
    """
    CREATE TABLE IF NOT EXISTS `tb_alarm_info` (
      `alarm_id` INT NOT NULL AUTO_INCREMENT,
      `user_num` INT NOT NULL COMMENT '?? ?? ?? ?? ??',
      `prediction_id` INT NOT NULL,
      `alarm_content` TEXT NOT NULL,
      `alarm_type` VARCHAR(10) NOT NULL COMMENT '??? or ??',
      `alarm_create_at` DATETIME NOT NULL,
      `anomaly_score` TEXT NOT NULL,
      CONSTRAINT `PK_TB_ALARM_INFO` PRIMARY KEY (`alarm_id`),
      CONSTRAINT `FK_ALARM_USER`
        FOREIGN KEY (`user_num`) REFERENCES `tb_user_info` (`user_num`),
      CONSTRAINT `FK_ALARM_PREDICTION`
        FOREIGN KEY (`prediction_id`) REFERENCES `tb_prediction` (`prediction_id`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
    """,
    """
    CREATE TABLE IF NOT EXISTS `tb_daily_report` (
      `report_id` INT NOT NULL AUTO_INCREMENT,
      `alarm_id` INT NOT NULL,
      `report_created_at` DATETIME NOT NULL,
      `issue_content` VARCHAR(255) NOT NULL,
      `action_detail` TEXT NULL,
      CONSTRAINT `PK_TB_DAILY_REPORT` PRIMARY KEY (`report_id`),
      CONSTRAINT `FK_DAILY_REPORT_ALARM`
        FOREIGN KEY (`alarm_id`) REFERENCES `tb_alarm_info` (`alarm_id`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
    """,
]

## 4. Create schema

If `RECREATE_TABLES=True`, existing target tables are dropped and created again.

In [5]:
def recreate_schema():
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute('SET FOREIGN_KEY_CHECKS = 0')
            if RECREATE_TABLES:
                for sql in DROP_TABLES_SQL:
                    cur.execute(sql)
            for sql in CREATE_TABLES_SQL:
                cur.execute(sql)
            cur.execute('SET FOREIGN_KEY_CHECKS = 1')
        conn.commit()
    finally:
        conn.close()


recreate_schema()
print('Schema is ready')

Schema is ready


## 4-1. Empty existing table data

Run this after schema creation and before CSV loading. It removes leftover rows from failed or partial previous loads.


In [6]:
conn = get_conn()
try:
    with conn.cursor() as cur:
        cur.execute('SET FOREIGN_KEY_CHECKS = 0')
        for table in reversed(LOAD_ORDER):
            cur.execute(f'TRUNCATE TABLE `{table}`')
        cur.execute('SET FOREIGN_KEY_CHECKS = 1')
    conn.commit()
finally:
    conn.close()

print('all tables are empty')


all tables are empty


## 5. Check CSV files

This step confirms that all expected CSV files exist and that each header matches the table definition.

In [7]:
def count_csv_rows(path: Path) -> int:
    with path.open('rb') as f:
        return max(sum(1 for _ in f) - 1, 0)


missing = [str(path) for path in CSV_FILES.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing CSV files:\n' + '\n'.join(missing))

CSV_ROW_COUNTS = {}
csv_check_rows = []

for table in LOAD_ORDER:
    path = CSV_FILES[table]
    csv_cols = list(pd.read_csv(path, nrows=0, encoding='utf-8-sig').columns)
    expected_cols = TABLE_COLUMNS[table]
    if csv_cols != expected_cols:
        raise ValueError(f'{table} column mismatch\nCSV: {csv_cols}\nEXPECTED: {expected_cols}')

    row_count = count_csv_rows(path)
    CSV_ROW_COUNTS[table] = row_count
    csv_check_rows.append({'table': table, 'csv_path': str(path), 'csv_rows': row_count})

csv_summary_df = pd.DataFrame(csv_check_rows)
display(csv_summary_df)

,table,csv_path,csv_rows
0,tb_power_station,C:\Users\Owner\Final_Project\final_project\dal...,5
1,tb_power_plant,C:\Users\Owner\Final_Project\final_project\dal...,45
2,tb_part,C:\Users\Owner\Final_Project\final_project\dal...,225
3,tb_user_info,C:\Users\Owner\Final_Project\final_project\dal...,2
4,tb_model_info,C:\Users\Owner\Final_Project\final_project\dal...,5
5,tb_mac_a_sensor,C:\Users\Owner\Final_Project\final_project\dal...,1578241
6,tb_mac_b_sensor,C:\Users\Owner\Final_Project\final_project\dal...,1578241
7,tb_bac_sensor,C:\Users\Owner\Final_Project\final_project\dal...,1578241
8,tb_dgan_sensor,C:\Users\Owner\Final_Project\final_project\dal...,1578241
9,tb_vhp_sensor,C:\Users\Owner\Final_Project\final_project\dal...,1578241


## 6. Load CSV files into MySQL

Tables are loaded in parent-to-child order so foreign keys can be validated.

In [8]:
def detect_line_ending(path: Path) -> str:
    sample = path.read_bytes()[:8192]
    return '\\r\\n' if b'\r\n' in sample else '\\n'


def get_insert_columns(table: str):
    """Return DB columns to insert.

    Sensor PK columns stay in the table and CSV, but are not inserted from CSV.
    MySQL assigns them with AUTO_INCREMENT. This avoids repeated-run and
    partial-load primary-key collisions while preserving the ERD table shape.
    """
    columns = TABLE_COLUMNS[table]
    if table in SENSOR_TABLES:
        return columns[1:]
    return columns


def get_load_data_columns_sql(table: str):
    """Build LOAD DATA column mapping.

    For sensor CSVs, the first CSV field is read into @skip_pk and ignored.
    The remaining fields are inserted into real table columns.
    """
    pieces = []
    for index, column in enumerate(TABLE_COLUMNS[table]):
        if table in SENSOR_TABLES and index == 0:
            pieces.append('@skip_pk')
        else:
            pieces.append(f'`{column}`')
    return ', '.join(pieces)


def get_db_row_count(table: str) -> int:
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute(f'SELECT COUNT(*) FROM `{table}`')
            return cur.fetchone()[0]
    finally:
        conn.close()


def empty_table(table: str):
    conn = get_conn(autocommit=True)
    try:
        with conn.cursor() as cur:
            cur.execute('SET FOREIGN_KEY_CHECKS = 0')
            cur.execute(f'TRUNCATE TABLE `{table}`')
            cur.execute(f'DELETE FROM `{table}`')
            cur.execute('SET FOREIGN_KEY_CHECKS = 1')
            cur.execute(f'SELECT COUNT(*) FROM `{table}`')
            remaining_rows = cur.fetchone()[0]
        if remaining_rows != 0:
            raise RuntimeError(f'{table}: table was not emptied before loading. remaining_rows={remaining_rows}')
    finally:
        conn.close()


def truncate_all_tables():
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute('SET FOREIGN_KEY_CHECKS = 0')
            for table in reversed(LOAD_ORDER):
                cur.execute(f'TRUNCATE TABLE `{table}`')
            cur.execute('SET FOREIGN_KEY_CHECKS = 1')
        conn.commit()
    finally:
        conn.close()


def load_csv_with_load_data(table: str, path: Path):
    col_sql = get_load_data_columns_sql(table)
    line_ending = detect_line_ending(path)
    sql = f"""
        LOAD DATA LOCAL INFILE %s
        INTO TABLE `{table}`
        CHARACTER SET utf8mb4
        FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"'
        LINES TERMINATED BY '{line_ending}'
        IGNORE 1 LINES
        ({col_sql})
    """
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute(sql, (path.as_posix(),))
        conn.commit()
    finally:
        conn.close()


def load_csv_with_chunks(table: str, path: Path, chunksize=CHUNKSIZE):
    insert_columns = get_insert_columns(table)
    placeholders = ', '.join(['%s'] * len(insert_columns))
    col_sql = ', '.join(f'`{col}`' for col in insert_columns)
    sql = f'INSERT INTO `{table}` ({col_sql}) VALUES ({placeholders})'

    total = 0
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            for chunk in pd.read_csv(path, chunksize=chunksize, low_memory=False, encoding='utf-8-sig'):
                chunk = chunk[insert_columns]
                chunk = chunk.astype(object).where(pd.notna(chunk), None)
                rows = list(chunk.itertuples(index=False, name=None))
                cur.executemany(sql, rows)
                conn.commit()
                total += len(rows)
                print(f'  {table}: {total:,} rows')
    finally:
        conn.close()


def load_one_table(table: str):
    path = CSV_FILES[table]
    expected_rows = CSV_ROW_COUNTS[table]
    start = time.time()

    empty_table(table)

    # Sensor CSVs are large but simple. Use pandas chunks for them because
    # MySQL LOAD DATA can occasionally over-read rows depending on local
    # infile and line-ending settings. Chunk loading gives exact row counts.
    use_load_data_for_table = USE_LOAD_DATA and table not in SENSOR_TABLES

    loaded_by_load_data = False
    if use_load_data_for_table:
        try:
            load_csv_with_load_data(table, path)
            loaded_by_load_data = True
        except pymysql.MySQLError as exc:
            print(f'{table}: LOAD DATA failed ({exc}). Falling back to chunk inserts.')

    if not loaded_by_load_data:
        empty_table(table)
        load_csv_with_chunks(table, path)

    db_rows = get_db_row_count(table)
    if db_rows != expected_rows:
        if loaded_by_load_data:
            print(f'{table}: row count mismatch after LOAD DATA. Retrying with chunk inserts.')
            empty_table(table)
            load_csv_with_chunks(table, path)
            db_rows = get_db_row_count(table)

    if db_rows != expected_rows:
        raise AssertionError(f'{table}: db_rows={db_rows}, csv_rows={expected_rows}')

    elapsed = time.time() - start
    print(f'{table} loaded: {db_rows:,} rows, {elapsed:.1f} sec')


truncate_all_tables()
print('Existing data truncated')

for table in LOAD_ORDER:
    load_one_table(table)

print('All CSV files loaded')

Existing data truncated
tb_power_station: LOAD DATA failed ((3948, 'Loading local data is disabled; this must be enabled on both the client and server sides')). Falling back to chunk inserts.
  tb_power_station: 5 rows
tb_power_station loaded: 5 rows, 0.1 sec
tb_power_plant: LOAD DATA failed ((3948, 'Loading local data is disabled; this must be enabled on both the client and server sides')). Falling back to chunk inserts.
  tb_power_plant: 45 rows
tb_power_plant loaded: 45 rows, 0.1 sec
tb_part: LOAD DATA failed ((3948, 'Loading local data is disabled; this must be enabled on both the client and server sides')). Falling back to chunk inserts.
  tb_part: 225 rows
tb_part loaded: 225 rows, 0.1 sec
tb_user_info: LOAD DATA failed ((3948, 'Loading local data is disabled; this must be enabled on both the client and server sides')). Falling back to chunk inserts.
  tb_user_info: 2 rows
tb_user_info loaded: 2 rows, 0.1 sec
tb_model_info: LOAD DATA failed ((3948, 'Loading local data is disabled

## 7. Confirm inserted row counts

Every `match` value should be `True`.

In [9]:
count_rows = []
for table in LOAD_ORDER:
    db_rows = get_db_row_count(table)
    csv_rows = CSV_ROW_COUNTS[table]
    count_rows.append({
        'table': table,
        'csv_rows': csv_rows,
        'mysql_rows': db_rows,
        'match': csv_rows == db_rows,
    })

row_count_df = pd.DataFrame(count_rows)
display(row_count_df)

if not row_count_df['match'].all():
    raise AssertionError('Some MySQL row counts do not match CSV row counts.')

,table,csv_rows,mysql_rows,match
0,tb_power_station,5,5,True
1,tb_power_plant,45,45,True
2,tb_part,225,225,True
3,tb_user_info,2,2,True
4,tb_model_info,5,5,True
5,tb_mac_a_sensor,1578241,1578241,True
6,tb_mac_b_sensor,1578241,1578241,True
7,tb_bac_sensor,1578241,1578241,True
8,tb_dgan_sensor,1578241,1578241,True
9,tb_vhp_sensor,1578241,1578241,True


## 8. Check foreign keys and sensor duplicates

Every `bad_rows` and `duplicate_keys` value should be `0`.

In [10]:
def fetch_df(sql, params=None):
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            rows = cur.fetchall()
            columns = [col[0] for col in cur.description] if cur.description else []
            return pd.DataFrame(rows, columns=columns)
    finally:
        conn.close()


fk_checks = {
    'tb_power_plant.station_id -> tb_power_station.station_id': """
        SELECT COUNT(*) AS bad_rows
        FROM tb_power_plant c
        LEFT JOIN tb_power_station p ON c.station_id = p.station_id
        WHERE p.station_id IS NULL
    """,
    'tb_part.plant_id -> tb_power_plant.plant_id': """
        SELECT COUNT(*) AS bad_rows
        FROM tb_part c
        LEFT JOIN tb_power_plant p ON c.plant_id = p.plant_id
        WHERE p.plant_id IS NULL
    """,
    'tb_model_info.part_id -> tb_part.part_id': """
        SELECT COUNT(*) AS bad_rows
        FROM tb_model_info c
        LEFT JOIN tb_part p ON c.part_id = p.part_id
        WHERE p.part_id IS NULL
    """,
    'tb_prediction.model_id -> tb_model_info.model_id': """
        SELECT COUNT(*) AS bad_rows
        FROM tb_prediction c
        LEFT JOIN tb_model_info p ON c.model_id = p.model_id
        WHERE p.model_id IS NULL
    """,
    'tb_alarm_info.user_num -> tb_user_info.user_num': """
        SELECT COUNT(*) AS bad_rows
        FROM tb_alarm_info c
        LEFT JOIN tb_user_info p ON c.user_num = p.user_num
        WHERE p.user_num IS NULL
    """,
    'tb_alarm_info.prediction_id -> tb_prediction.prediction_id': """
        SELECT COUNT(*) AS bad_rows
        FROM tb_alarm_info c
        LEFT JOIN tb_prediction p ON c.prediction_id = p.prediction_id
        WHERE p.prediction_id IS NULL
    """,
    'tb_daily_report.alarm_id -> tb_alarm_info.alarm_id': """
        SELECT COUNT(*) AS bad_rows
        FROM tb_daily_report c
        LEFT JOIN tb_alarm_info p ON c.alarm_id = p.alarm_id
        WHERE p.alarm_id IS NULL
    """,
}

for sensor_table in SENSOR_TABLES:
    fk_checks[f'{sensor_table}.part_id -> tb_part.part_id'] = f"""
        SELECT COUNT(*) AS bad_rows
        FROM `{sensor_table}` c
        LEFT JOIN tb_part p ON c.part_id = p.part_id
        WHERE p.part_id IS NULL
    """

fk_rows = []
for check_name, sql in fk_checks.items():
    bad_rows = int(fetch_df(sql).iloc[0, 0])
    fk_rows.append({'check': check_name, 'bad_rows': bad_rows})

fk_df = pd.DataFrame(fk_rows)
display(fk_df)

duplicate_rows = []
for table in SENSOR_TABLES:
    sql = f"""
        SELECT COUNT(*) AS duplicate_keys
        FROM (
            SELECT part_id, measured_at, COUNT(*) AS cnt
            FROM `{table}`
            GROUP BY part_id, measured_at
            HAVING COUNT(*) > 1
        ) dup
    """
    duplicate_keys = int(fetch_df(sql).iloc[0, 0])
    duplicate_rows.append({'table': table, 'duplicate_keys': duplicate_keys})

duplicate_df = pd.DataFrame(duplicate_rows)
display(duplicate_df)

if fk_df['bad_rows'].sum() != 0:
    raise AssertionError('Foreign key check failed.')
if duplicate_df['duplicate_keys'].sum() != 0:
    raise AssertionError('Sensor duplicate key check failed.')

,check,bad_rows
0,tb_power_plant.station_id -> tb_power_station....,0
1,tb_part.plant_id -> tb_power_plant.plant_id,0
2,tb_model_info.part_id -> tb_part.part_id,0
3,tb_prediction.model_id -> tb_model_info.model_id,0
4,tb_alarm_info.user_num -> tb_user_info.user_num,0
5,tb_alarm_info.prediction_id -> tb_prediction.p...,0
6,tb_daily_report.alarm_id -> tb_alarm_info.alar...,0
7,tb_mac_a_sensor.part_id -> tb_part.part_id,0
8,tb_mac_b_sensor.part_id -> tb_part.part_id,0
9,tb_bac_sensor.part_id -> tb_part.part_id,0


,table,duplicate_keys
0,tb_mac_a_sensor,0
1,tb_mac_b_sensor,0
2,tb_bac_sensor,0
3,tb_dgan_sensor,0
4,tb_vhp_sensor,0


## 9. Show loaded table samples

This confirms the inserted data with `SELECT *` samples from each table.

In [11]:
for table in LOAD_ORDER:
    print('\n' + '=' * 100)
    print(f'TABLE: {table}')
    print(f'rows: {get_db_row_count(table):,}')
    display(fetch_df(f'SELECT * FROM `{table}` LIMIT {SAMPLE_ROWS}'))


TABLE: tb_power_station
rows: 5


,station_id,station_name,station_lat,station_lon,station_add,station_capacity
0,1,태안발전본부,36.905568,126.234627,충청남도 태안군 원북면 발전로 457,6504.5
1,2,서인천발전본부,37.536300,126.602700,인천광역시 서구 장도로 57,1861.8
2,3,평택발전본부,37.005802,126.798410,경기도 평택시 포승읍 남양만로 175-2,871.4
3,4,군산발전본부,35.983600,126.730700,전북특별자치도 군산시 구암3.1로 91-5,719.4
4,5,김포발전본부,37.600220,126.587781,경기도 김포시 양촌읍 학운산단1로 76 학운2산업단지,510.0



TABLE: tb_power_plant
rows: 45


,plant_id,station_id,plant_name,fuel_type,plant_status
0,1,1,태안#1,유연탄(석탄),발전종료
1,2,1,태안#2,유연탄(석탄),운영중
2,3,1,태안#3,유연탄(석탄),운영중
3,4,1,태안#4,유연탄(석탄),운영중
4,5,1,태안#5,유연탄(석탄),운영중



TABLE: tb_part
rows: 225


...

,prediction_id,model_id,anomaly_score,prediction_time,current_recon_error,NDE_temp_recon_error,DE_temp_recon_error,NDE_X_vib_recon_error,NDE_Y_vib_recon_error,DE_X_vib_recon_error,DE_Y_vib_recon_error
0,1,1,0.000442,2023-01-01 00:00:00,0.000442,0.000442,0.000442,0.000442,0.000442,0.000442,0.000442
1,2,1,0.000202,2023-01-01 00:01:00,0.000202,0.000202,0.000202,0.000202,0.000202,0.000202,0.000202
2,3,1,0.001043,2023-01-01 00:02:00,0.001043,0.001043,0.001043,0.001043,0.001043,0.001043,0.001043
3,4,1,0.000779,2023-01-01 00:03:00,0.000779,0.000779,0.000779,0.000779,0.000779,0.000779,0.000779
4,5,1,0.000069,2023-01-01 00:04:00,0.000069,0.000069,0.000069,0.000069,0.000069,0.000069,0.000069



TABLE: tb_alarm_info
rows: 37,094


,alarm_id,user_num,prediction_id,alarm_content,alarm_type,alarm_create_at,anomaly_score
0,1,1,349,IGCC_MAC_A anomaly_score=0.00369653 threshold=...,email,2023-01-01 05:49:00,0.0036965306
1,2,1,349,IGCC_MAC_A anomaly_score=0.00369653 threshold=...,slack,2023-01-01 05:49:00,0.0036965306
2,3,1,589,IGCC_MAC_A anomaly_score=0.00215758 threshold=...,email,2023-01-01 09:49:00,0.0021575845
3,4,1,589,IGCC_MAC_A anomaly_score=0.00215758 threshold=...,slack,2023-01-01 09:49:00,0.0021575845
4,5,1,1071,IGCC_MAC_A anomaly_score=0.00317157 threshold=...,email,2023-01-01 17:51:00,0.0031715657



TABLE: tb_daily_report
rows: 5


,report_id,alarm_id,report_created_at,issue_content,action_detail
0,1,1,2026-05-20 09:10:00,NDE bearing temperature exceeded threshold,Checked cooling fan and scheduled bearing insp...
1,2,2,2026-05-20 10:25:00,DE X-axis vibration increased,Reduced load temporarily and requested vibrati...
2,3,3,2026-05-20 11:40:00,Motor current anomaly detected,Verified power supply condition and monitored ...
3,4,4,2026-05-20 13:15:00,NDE Y-axis vibration warning,Inspected coupling alignment and tightened mou...
4,5,5,2026-05-20 15:30:00,DE bearing temperature warning,Cleaned ventilation area and added follow-up t...


## 10. Show actual MySQL columns

Use this if you want to compare the created table schema with the CSV headers.

In [12]:
for table in LOAD_ORDER:
    print('\n' + '=' * 100)
    print(f'COLUMNS: {table}')
    display(fetch_df(f'SHOW COLUMNS FROM `{table}`'))


COLUMNS: tb_power_station


,Field,Type,Null,Key,Default,Extra
0,station_id,int,NO,PRI,None,auto_increment
1,station_name,varchar(255),NO,,None,
2,station_lat,double,YES,,None,
3,station_lon,double,YES,,None,
4,station_add,varchar(255),NO,,None,
5,station_capacity,double,NO,,None,



COLUMNS: tb_power_plant


,Field,Type,Null,Key,Default,Extra
0,plant_id,int,NO,PRI,None,auto_increment
1,station_id,int,NO,MUL,None,
2,plant_name,varchar(255),NO,,None,
3,fuel_type,varchar(255),NO,,None,
4,plant_status,varchar(255),NO,,None,



COLUMNS: tb_part


...

,Field,Type,Null,Key,Default,Extra
0,prediction_id,int,NO,PRI,None,auto_increment
1,model_id,int,NO,MUL,None,
2,anomaly_score,double,NO,,None,
3,prediction_time,datetime,NO,,None,
4,current_recon_error,double,NO,,None,
5,NDE_temp_recon_error,double,NO,,None,
6,DE_temp_recon_error,double,NO,,None,
7,NDE_X_vib_recon_error,double,NO,,None,
8,NDE_Y_vib_recon_error,double,NO,,None,
9,DE_X_vib_recon_error,double,NO,,None,



COLUMNS: tb_alarm_info


,Field,Type,Null,Key,Default,Extra
0,alarm_id,int,NO,PRI,None,auto_increment
1,user_num,int,NO,MUL,None,
2,prediction_id,int,NO,MUL,None,
3,alarm_content,text,NO,,None,
4,alarm_type,varchar(10),NO,,None,
5,alarm_create_at,datetime,NO,,None,
6,anomaly_score,text,NO,,None,



COLUMNS: tb_daily_report


,Field,Type,Null,Key,Default,Extra
0,report_id,int,NO,PRI,None,auto_increment
1,alarm_id,int,NO,MUL,None,
2,report_created_at,datetime,NO,,None,
3,issue_content,varchar(255),NO,,None,
4,action_detail,text,YES,,None,
